In [ ]:
"""
GEN5 H100 STRESS SWEEP — CASE WESTERN RESERVE BEARING FAULT DATA
================================================================
Real industrial data. Gen5 only. Three sizes × 20 seeds. Full logging,
full visual dashboards.

What it does:
  1. Downloads CWRU bearing fault dataset (4 conditions: normal, IR, OR, ball)
  2. Builds windows from raw vibration signals
  3. Trains Gen5 at H=32, H=64, H=128 across 20 seeds each (60 runs)
  4. Logs full training trajectories: loss, spectral radius, learnable params,
     alpha mixing, field accumulator, p-extension norms
  5. Produces one PDF dashboard per run + one summary dashboard
  6. Saves checkpoints every run (crash-safe)

Run:
    pip install torch numpy scipy matplotlib requests
    python3 gen5_h100_stress.py

GPU: detects CUDA automatically, uses it if available. Falls back to CPU.
Wall: ~4-6 hours on H100. ~50+ hours on CPU (not recommended).

OUTPUT:
    ./gen5_h100_out/data/         — downloaded .mat files
    ./gen5_h100_out/checkpoints/  — model state dicts per run
    ./gen5_h100_out/results.json  — all numeric data
    ./gen5_h100_out/per_run/      — 60 individual PDF dashboards
    ./gen5_h100_out/summary.pdf   — cross-run comparison dashboard
"""

import os, math, json, time, hashlib, urllib.request, pathlib, traceback
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ─────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────
OUT_DIR = pathlib.Path("./gen5_h100_out")
DATA_DIR = OUT_DIR / "data"
CKPT_DIR = OUT_DIR / "checkpoints"
RUN_PDF_DIR = OUT_DIR / "per_run"
for d in [OUT_DIR, DATA_DIR, CKPT_DIR, RUN_PDF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] device = {DEVICE}")
if DEVICE.type == "cuda":
    print(f"[init] gpu   = {torch.cuda.get_device_name(0)}")
    print(f"[init] mem   = {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

SIZES = [32, 64, 128]
SEEDS = list(range(20))
TRAIN_STEPS = 2000
LOG_EVERY = 50
BATCH = 32
WINDOW = 1024              # samples per window
LR = 3e-3

# ─────────────────────────────────────────────────────────────────────
# DATASET: Case Western Reserve Bearing Fault Data
# CWRU hosts .mat files. Mirror via direct URLs.
# ─────────────────────────────────────────────────────────────────────
# 12kHz drive-end bearing fault data, motor load 1HP, ~1750 rpm
# Normal baseline + three fault types, each at 0.007" fault diameter
CWRU_FILES = {
    "normal":   "https://engineering.case.edu/sites/default/files/97.mat",
    "inner":    "https://engineering.case.edu/sites/default/files/105.mat",
    "outer":    "https://engineering.case.edu/sites/default/files/130.mat",
    "ball":     "https://engineering.case.edu/sites/default/files/118.mat",
}
LABELS = list(CWRU_FILES.keys())   # 0=normal, 1=inner, 2=outer, 3=ball

def download_cwru():
    for name, url in CWRU_FILES.items():
        path = DATA_DIR / f"{name}.mat"
        if path.exists() and path.stat().st_size > 1000:
            print(f"[data] {name}: cached")
            continue
        print(f"[data] {name}: downloading from {url}")
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as r:
                path.write_bytes(r.read())
            print(f"[data] {name}: {path.stat().st_size//1024} KB")
        except Exception as e:
            print(f"[data] FAILED {name}: {e}")
            print(f"[data] If download fails, manually download {url} -> {path}")
            raise

def load_cwru():
    from scipy.io import loadmat
    signals = {}
    for name in LABELS:
        path = DATA_DIR / f"{name}.mat"
        mat = loadmat(str(path))
        # CWRU files have a key like 'X097_DE_time' for drive-end vibration
        # Find the DE_time channel automatically
        de_key = None
        for k in mat.keys():
            if "DE_time" in k:
                de_key = k
                break
        if de_key is None:
            # fallback: largest 1-D-ish array
            best_size = 0
            for k, v in mat.items():
                if not k.startswith("_") and isinstance(v, np.ndarray) and v.size > best_size:
                    de_key = k
                    best_size = v.size
        sig = mat[de_key].flatten().astype(np.float32)
        signals[name] = sig
        print(f"[data] {name}: {len(sig)} samples ({len(sig)/12000:.1f}s @ 12kHz)")
    return signals

def make_windows(signals, window=WINDOW, stride=512):
    """Build (signal, label) pairs. Standardize per-signal to zero mean unit std."""
    Xs, Ys = [], []
    for label, name in enumerate(LABELS):
        sig = signals[name]
        sig = (sig - sig.mean()) / (sig.std() + 1e-8)
        # slide windows
        for start in range(0, len(sig) - window, stride):
            Xs.append(sig[start:start+window])
            Ys.append(label)
    X = np.stack(Xs)          # (N, window)
    Y = np.array(Ys)
    # shuffle and split 80/20
    rng = np.random.default_rng(42)
    idx = rng.permutation(len(X))
    X = X[idx]; Y = Y[idx]
    n_train = int(0.8 * len(X))
    X_tr, Y_tr = X[:n_train], Y[:n_train]
    X_te, Y_te = X[n_train:], Y[n_train:]
    print(f"[data] windows: {len(X_tr)} train, {len(X_te)} test, {len(LABELS)} classes")
    return X_tr, Y_tr, X_te, Y_te

# ─────────────────────────────────────────────────────────────────────
# GEN5 MODEL — eight-layer composed substrate
# Adapted for classification head (4 classes) from time-series window
# ─────────────────────────────────────────────────────────────────────
def sphere_proj(h, r=1.0):
    nrm = h.norm(dim=-1, keepdim=True).clamp_min(1e-6)
    return torch.tanh(nrm/r)*r*(h/nrm)

def cube_proj(h, r=1.0):
    return torch.tanh(h/r)*r

def smooth_max(h, beta=8.0):
    sat = h.abs().clamp(0, 5)
    return (1.0/beta)*torch.logsumexp(beta*sat, dim=-1, keepdim=True)

class Gen5(nn.Module):
    def __init__(self, H, in_dim=1, n_classes=4, r_outer=1.0, r_inner=0.65, R_dodec=1.2):
        super().__init__()
        self.H = H
        self.r_outer, self.r_inner, self.R_dodec = r_outer, r_inner, R_dodec
        self.inp = nn.Linear(in_dim, H)
        self.W_s  = nn.Linear(H, H, bias=False)
        self.W_px = nn.Linear(H, H, bias=False)
        self.W_py = nn.Linear(H, H, bias=False)
        self.W_pz = nn.Linear(H, H, bias=False)
        self.couple   = nn.Parameter(torch.tensor(0.30))
        self.pi_dyn   = nn.Parameter(torch.tensor(4.0))
        self.phi_dyn  = nn.Parameter(torch.tensor(0.70))
        self.vesica_a = nn.Parameter(torch.tensor(0.40))
        self.vesica_b = nn.Parameter(torch.tensor(0.40))
        self.head = nn.Linear(H, n_classes)

    def _paired(self, h):
        sat = smooth_max(h)
        a = torch.sigmoid(self.pi_dyn*(sat - self.phi_dyn))
        sp = sphere_proj(h, self.r_inner)
        cu = cube_proj(h, self.r_outer)
        return (1.0-a)*sp + a*cu, a

    def _vesica(self, m):
        return -(-self.vesica_a*m + self.vesica_b*m**3)

    def _dodec(self, h):
        return torch.clamp(h, -self.R_dodec, self.R_dodec)

    def forward(self, x, return_trace=False):
        # x: (B, T) — single channel vibration window
        if x.dim() == 2:
            x = x.unsqueeze(-1)
        B, T, _ = x.shape
        s  = torch.zeros(B, self.H, device=x.device)
        px = torch.zeros_like(s); py = torch.zeros_like(s); pz = torch.zeros_like(s)

        field_trace = []
        alpha_trace = []
        for ti in range(T):
            u = self.inp(x[:, ti])
            px_raw = self.W_px(px) + u - self.couple*(px - s) + self._vesica(px)*0.1
            py_raw = self.W_py(py) + u - self.couple*(py - s) + self._vesica(py)*0.1
            pz_raw = self.W_pz(pz) + u - self.couple*(pz - s) + self._vesica(pz)*0.1
            px, a_x = self._paired(px_raw); px = self._dodec(px)
            py, a_y = self._paired(py_raw); py = self._dodec(py)
            pz, a_z = self._paired(pz_raw); pz = self._dodec(pz)
            s_raw = self.W_s(s) + (px+py+pz)/3
            s, a_s = self._paired(s_raw); s = self._dodec(s)
            if return_trace:
                with torch.no_grad():
                    field_trace.append(s.norm(dim=-1).cpu().numpy())
                    alpha_trace.append(((a_x+a_y+a_z+a_s)/4).mean().item())

        logits = self.head(s)
        if return_trace:
            return logits, np.array(field_trace), np.array(alpha_trace)
        return logits

    @staticmethod
    def recurrent_params(H): return 4*H*H

# ─────────────────────────────────────────────────────────────────────
# TRAINING WITH FULL LOGGING
# ─────────────────────────────────────────────────────────────────────
def spectral_radius(W):
    with torch.no_grad():
        evs = torch.linalg.eigvals(W.float()).abs()
        return float(evs.max().item())

def train_one(H, seed, X_tr, Y_tr, X_te, Y_te):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = Gen5(H=H).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    X_tr_t = torch.from_numpy(X_tr).float().to(DEVICE)
    Y_tr_t = torch.from_numpy(Y_tr).long().to(DEVICE)
    X_te_t = torch.from_numpy(X_te).float().to(DEVICE)
    Y_te_t = torch.from_numpy(Y_te).long().to(DEVICE)

    log = {
        'step': [], 'loss': [],
        'sr_Ws': [], 'sr_Wpx': [], 'sr_Wpy': [], 'sr_Wpz': [],
        'pi_dyn': [], 'phi_dyn': [], 'couple': [],
        'vesica_a': [], 'vesica_b': [],
        'test_acc': [],
    }

    t0 = time.time()
    n_tr = len(X_tr_t)
    for step in range(TRAIN_STEPS):
        idx = torch.randint(0, n_tr, (BATCH,), device=DEVICE)
        x, y = X_tr_t[idx], Y_tr_t[idx]
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()

        if step % LOG_EVERY == 0 or step == TRAIN_STEPS - 1:
            log['step'].append(step)
            log['loss'].append(float(loss.item()))
            log['sr_Ws'].append(spectral_radius(model.W_s.weight))
            log['sr_Wpx'].append(spectral_radius(model.W_px.weight))
            log['sr_Wpy'].append(spectral_radius(model.W_py.weight))
            log['sr_Wpz'].append(spectral_radius(model.W_pz.weight))
            log['pi_dyn'].append(float(model.pi_dyn.item()))
            log['phi_dyn'].append(float(model.phi_dyn.item()))
            log['couple'].append(float(model.couple.item()))
            log['vesica_a'].append(float(model.vesica_a.item()))
            log['vesica_b'].append(float(model.vesica_b.item()))
            # test acc on a subsample (fast)
            with torch.no_grad():
                model.eval()
                te_idx = torch.randint(0, len(X_te_t), (min(256, len(X_te_t)),), device=DEVICE)
                pred = model(X_te_t[te_idx]).argmax(-1)
                acc = float((pred == Y_te_t[te_idx]).float().mean().item())
                log['test_acc'].append(acc)
                model.train()

    wall = time.time() - t0

    # final full test eval + field traces by class
    model.eval()
    with torch.no_grad():
        # full test accuracy
        all_pred = []
        for i in range(0, len(X_te_t), 64):
            chunk = X_te_t[i:i+64]
            all_pred.append(model(chunk).argmax(-1).cpu().numpy())
        all_pred = np.concatenate(all_pred)
        final_acc = float((all_pred == Y_te).mean())

        # per-class field traces: one window per class, get the trace
        class_traces = {}
        class_alphas = {}
        for label, name in enumerate(LABELS):
            class_mask = (Y_te == label)
            if class_mask.sum() == 0: continue
            sample_idx = np.where(class_mask)[0][0]
            x = X_te_t[sample_idx:sample_idx+1]
            _, field, alpha = model(x, return_trace=True)
            class_traces[name] = field.flatten()
            class_alphas[name] = alpha.flatten()

        # final p-extension norms (mean over test set)
        x_sample = X_te_t[:64]
        s_state  = torch.zeros(64, model.H, device=DEVICE)
        px_state = torch.zeros_like(s_state); py_state = torch.zeros_like(s_state); pz_state = torch.zeros_like(s_state)
        x_in = x_sample.unsqueeze(-1) if x_sample.dim() == 2 else x_sample
        for ti in range(x_in.shape[1]):
            u = model.inp(x_in[:, ti])
            px_raw = model.W_px(px_state) + u - model.couple*(px_state - s_state) + model._vesica(px_state)*0.1
            py_raw = model.W_py(py_state) + u - model.couple*(py_state - s_state) + model._vesica(py_state)*0.1
            pz_raw = model.W_pz(pz_state) + u - model.couple*(pz_state - s_state) + model._vesica(pz_state)*0.1
            px_state, _ = model._paired(px_raw); px_state = model._dodec(px_state)
            py_state, _ = model._paired(py_raw); py_state = model._dodec(py_state)
            pz_state, _ = model._paired(pz_raw); pz_state = model._dodec(pz_state)
            s_raw = model.W_s(s_state) + (px_state+py_state+pz_state)/3
            s_state, _ = model._paired(s_raw); s_state = model._dodec(s_state)
        p_norms = {
            'px_norm_mean': float(px_state.norm(dim=-1).mean().item()),
            'py_norm_mean': float(py_state.norm(dim=-1).mean().item()),
            'pz_norm_mean': float(pz_state.norm(dim=-1).mean().item()),
            's_norm_mean':  float(s_state.norm(dim=-1).mean().item()),
        }
        # also field separation at final timestep, per class
        field_separation = {}
        for label, name in enumerate(LABELS):
            class_mask = (Y_te == label)
            if class_mask.sum() == 0: continue
            x = X_te_t[class_mask][:32]
            _, field, _ = model(x, return_trace=True)
            field_separation[name] = {
                'final_field_mean': float(field[-1].mean()),
                'final_field_std':  float(field[-1].std()),
            }

    result = {
        'H': H, 'seed': seed,
        'recurrent_params': Gen5.recurrent_params(H),
        'wall_seconds': wall,
        'final_test_acc': final_acc,
        'log': log,
        'class_traces': {k: v.tolist() for k, v in class_traces.items()},
        'class_alphas': {k: v.tolist() for k, v in class_alphas.items()},
        'p_norms': p_norms,
        'field_separation': field_separation,
    }

    # save checkpoint
    ckpt_path = CKPT_DIR / f"gen5_H{H}_seed{seed}.pt"
    torch.save({'model': model.state_dict(), 'result_summary': {
        'H': H, 'seed': seed, 'final_test_acc': final_acc,
        'wall_seconds': wall,
    }}, ckpt_path)

    return result

# ─────────────────────────────────────────────────────────────────────
# PER-RUN PDF DASHBOARD
# ─────────────────────────────────────────────────────────────────────
def plot_run(result, path):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.backends.backend_pdf import PdfPages

    H, seed = result['H'], result['seed']
    log = result['log']

    with PdfPages(path) as pdf:
        fig, axes = plt.subplots(3, 2, figsize=(11, 13))
        fig.suptitle(f"Gen5  H={H}  seed={seed}  test_acc={result['final_test_acc']:.3f}  "
                     f"wall={result['wall_seconds']:.1f}s",
                     fontsize=12, color='#b8941f', weight='bold')

        # 1. Loss curve + test acc
        ax = axes[0,0]
        ax.plot(log['step'], log['loss'], color='#b8941f', lw=1.4, label='train loss')
        ax2 = ax.twinx()
        ax2.plot(log['step'], log['test_acc'], color='#8eb4dc', lw=1.4, label='test acc')
        ax.set_xlabel('step'); ax.set_ylabel('loss', color='#b8941f')
        ax2.set_ylabel('test acc', color='#8eb4dc'); ax2.set_ylim(0, 1.05)
        ax.set_title('Training trajectory')
        ax.grid(alpha=0.3)

        # 2. Spectral radii of all four W matrices
        ax = axes[0,1]
        ax.plot(log['step'], log['sr_Ws'],  label='W_s',  color='#e8c558', lw=1.5)
        ax.plot(log['step'], log['sr_Wpx'], label='W_px', color='#e09060', lw=1.0)
        ax.plot(log['step'], log['sr_Wpy'], label='W_py', color='#8fd99a', lw=1.0)
        ax.plot(log['step'], log['sr_Wpz'], label='W_pz', color='#8eb4dc', lw=1.0)
        ax.axhline(4/math.pi, color='#776a52', ls='--', lw=0.8, label='4/pi')
        ax.axhline(8/math.pi**2, color='#776a52', ls=':', lw=0.8, label='8/pi^2')
        ax.set_xlabel('step'); ax.set_ylabel('spectral radius')
        ax.set_title('Spectral radii — recurrent matrices')
        ax.legend(loc='best', fontsize=8); ax.grid(alpha=0.3)

        # 3. Learnable scalars drift
        ax = axes[1,0]
        ax.plot(log['step'], log['pi_dyn'],   label='pi_dyn',  color='#e8c558', lw=1.5)
        ax.plot(log['step'], log['phi_dyn'],  label='phi_dyn', color='#e09060', lw=1.5)
        ax.plot(log['step'], log['couple'],   label='couple',  color='#8fd99a', lw=1.5)
        ax.plot(log['step'], log['vesica_a'], label='vesica_a',color='#8eb4dc', lw=1.0)
        ax.plot(log['step'], log['vesica_b'], label='vesica_b',color='#a78ed6', lw=1.0)
        ax.set_xlabel('step'); ax.set_ylabel('value')
        ax.set_title('Learnable edge/saddle parameters')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

        # 4. Field accumulator over signal, per class
        ax = axes[1,1]
        colors = {'normal':'#8fd99a', 'inner':'#e09060', 'outer':'#e8c558', 'ball':'#8eb4dc'}
        for name, trace in result['class_traces'].items():
            ax.plot(trace, label=name, color=colors.get(name, 'gray'), lw=1.2)
        ax.set_xlabel('signal timestep')
        ax.set_ylabel('|s| field magnitude')
        ax.set_title('L0 field accumulator per class')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

        # 5. Alpha mixing (L5 sphere/cube) per class
        ax = axes[2,0]
        for name, alpha in result['class_alphas'].items():
            ax.plot(alpha, label=name, color=colors.get(name, 'gray'), lw=1.2)
        ax.axhline(0.5, color='#776a52', ls='--', lw=0.8)
        ax.set_xlabel('signal timestep')
        ax.set_ylabel('alpha (0=sphere, 1=cube)')
        ax.set_title('L5 paired-bound mixing weight')
        ax.set_ylim(-0.05, 1.05)
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

        # 6. Field separation final values, box plot
        ax = axes[2,1]
        names = list(result['field_separation'].keys())
        means = [result['field_separation'][n]['final_field_mean'] for n in names]
        stds  = [result['field_separation'][n]['final_field_std'] for n in names]
        bar_colors = [colors.get(n, 'gray') for n in names]
        ax.bar(names, means, yerr=stds, color=bar_colors, edgecolor='#6e5614', capsize=4)
        ax.set_ylabel('final |s| field')
        ax.set_title('Field separation by class (final timestep)')
        ax.grid(alpha=0.3, axis='y')

        plt.tight_layout()
        pdf.savefig(fig, facecolor='white')
        plt.close(fig)

# ─────────────────────────────────────────────────────────────────────
# SUMMARY PDF — across all runs
# ─────────────────────────────────────────────────────────────────────
def plot_summary(all_results, path):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.backends.backend_pdf import PdfPages

    with PdfPages(path) as pdf:
        # ── page 1: test accuracy vs size, all seeds ──
        fig, axes = plt.subplots(2, 2, figsize=(11, 9))
        fig.suptitle("Gen5 Stress Sweep — CWRU Bearing Fault Data — Summary",
                     fontsize=13, color='#b8941f', weight='bold')

        ax = axes[0,0]
        for H in SIZES:
            accs = [r['final_test_acc'] for r in all_results if r['H']==H]
            ax.scatter([H]*len(accs), accs, color='#b8941f', s=22, alpha=0.55)
            ax.scatter([H], [np.mean(accs)], color='#e8c558', s=120, marker='_', lw=3)
        ax.set_xlabel('hidden size H'); ax.set_ylabel('final test accuracy')
        ax.set_title('Accuracy vs size (each dot = one seed)')
        ax.set_xscale('log'); ax.grid(alpha=0.3)

        # ── final spectral radius of W_s per size ──
        ax = axes[0,1]
        for H in SIZES:
            srs = [r['log']['sr_Ws'][-1] for r in all_results if r['H']==H]
            ax.scatter([H]*len(srs), srs, color='#b8941f', s=22, alpha=0.55)
            ax.scatter([H], [np.mean(srs)], color='#e8c558', s=120, marker='_', lw=3)
        ax.axhline(4/math.pi, color='#776a52', ls='--', lw=0.8, label='4/pi')
        ax.set_xlabel('hidden size H'); ax.set_ylabel('final spectral radius W_s')
        ax.set_title('Lattice read: final W_s spectral radius')
        ax.set_xscale('log'); ax.legend(); ax.grid(alpha=0.3)

        # ── pi_dyn final values per size ──
        ax = axes[1,0]
        for H in SIZES:
            pis = [r['log']['pi_dyn'][-1] for r in all_results if r['H']==H]
            phs = [r['log']['phi_dyn'][-1] for r in all_results if r['H']==H]
            ax.scatter([H]*len(pis), pis, color='#e8c558', s=22, alpha=0.55, label='pi_dyn' if H==SIZES[0] else None)
            ax.scatter([H]*len(phs), phs, color='#e09060', s=22, alpha=0.55, label='phi_dyn' if H==SIZES[0] else None)
        ax.axhline(4.0, color='#e8c558', ls=':', lw=0.8, alpha=0.6)
        ax.axhline(0.7, color='#e09060', ls=':', lw=0.8, alpha=0.6)
        ax.set_xlabel('hidden size H'); ax.set_ylabel('learned value')
        ax.set_title('Edge parameters: drift from initialization (dotted)')
        ax.set_xscale('log'); ax.legend(); ax.grid(alpha=0.3)

        # ── wall time per size ──
        ax = axes[1,1]
        for H in SIZES:
            walls = [r['wall_seconds'] for r in all_results if r['H']==H]
            ax.scatter([H]*len(walls), walls, color='#8fd99a', s=22, alpha=0.55)
            ax.scatter([H], [np.mean(walls)], color='#e8c558', s=120, marker='_', lw=3)
        ax.set_xlabel('hidden size H'); ax.set_ylabel('wall seconds')
        ax.set_title(f'Training wall time ({TRAIN_STEPS} steps)')
        ax.set_xscale('log'); ax.set_yscale('log'); ax.grid(alpha=0.3)

        plt.tight_layout()
        pdf.savefig(fig, facecolor='white')
        plt.close(fig)

        # ── page 2: loss curves overlaid ──
        fig, axes = plt.subplots(len(SIZES), 1, figsize=(10, 4*len(SIZES)))
        if len(SIZES) == 1: axes = [axes]
        for ax, H in zip(axes, SIZES):
            for r in all_results:
                if r['H'] != H: continue
                ax.plot(r['log']['step'], r['log']['loss'], color='#b8941f', lw=0.7, alpha=0.5)
            ax.set_title(f'Loss curves — H={H} (all {len(SEEDS)} seeds)')
            ax.set_xlabel('step'); ax.set_ylabel('cross-entropy loss')
            ax.grid(alpha=0.3)
        plt.tight_layout()
        pdf.savefig(fig, facecolor='white')
        plt.close(fig)

        # ── page 3: field separation across all runs ──
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))
        ax.set_title("Field separation by class — all runs aggregated")
        all_by_class = {n: [] for n in LABELS}
        for r in all_results:
            for n, stats in r['field_separation'].items():
                all_by_class[n].append(stats['final_field_mean'])
        positions = list(range(len(LABELS)))
        bp = ax.boxplot([all_by_class[n] for n in LABELS], positions=positions,
                        widths=0.5, patch_artist=True)
        for patch, name in zip(bp['boxes'], LABELS):
            patch.set_facecolor({'normal':'#8fd99a', 'inner':'#e09060',
                                 'outer':'#e8c558', 'ball':'#8eb4dc'}[name])
            patch.set_alpha(0.7)
        ax.set_xticks(positions); ax.set_xticklabels(LABELS)
        ax.set_ylabel('final |s| field')
        ax.grid(alpha=0.3, axis='y')
        plt.tight_layout()
        pdf.savefig(fig, facecolor='white')
        plt.close(fig)

# ─────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────
def main():
    print("\n" + "="*70)
    print("GEN5 H100 STRESS SWEEP — CWRU BEARING FAULT")
    print(f"Sizes: {SIZES}  ·  Seeds: {len(SEEDS)}  ·  Steps: {TRAIN_STEPS}")
    print(f"Total runs: {len(SIZES) * len(SEEDS)}")
    print("="*70)

    # 1. data
    download_cwru()
    signals = load_cwru()
    X_tr, Y_tr, X_te, Y_te = make_windows(signals)

    # 2. sweep
    all_results = []
    results_path = OUT_DIR / "results.json"
    # resume support
    if results_path.exists():
        all_results = json.loads(results_path.read_text())
        done = {(r['H'], r['seed']) for r in all_results}
        print(f"[resume] found {len(all_results)} completed runs; skipping those")
    else:
        done = set()

    total = len(SIZES) * len(SEEDS)
    run_idx = 0
    overall_t0 = time.time()
    for H in SIZES:
        for seed in SEEDS:
            run_idx += 1
            if (H, seed) in done:
                print(f"[{run_idx}/{total}] SKIP  H={H} seed={seed}")
                continue
            print(f"[{run_idx}/{total}] RUN   H={H} seed={seed}  ", end='', flush=True)
            try:
                result = train_one(H, seed, X_tr, Y_tr, X_te, Y_te)
                all_results.append(result)
                # persist after every run (crash-safe)
                results_path.write_text(json.dumps(all_results, indent=2))
                # per-run pdf
                pdf_path = RUN_PDF_DIR / f"gen5_H{H}_seed{seed}.pdf"
                plot_run(result, pdf_path)
                print(f"acc={result['final_test_acc']:.3f}  "
                      f"wall={result['wall_seconds']:.1f}s")
            except Exception as e:
                print(f"FAILED: {e}")
                traceback.print_exc()
                # write a marker so we don't loop on it
                continue

    print(f"\n[done] sweep wall: {(time.time()-overall_t0)/60:.1f} min")

    # 3. summary dashboard
    print("[plot] generating summary PDF...")
    plot_summary(all_results, OUT_DIR / "summary.pdf")
    print(f"[plot] saved: {OUT_DIR / 'summary.pdf'}")

    print("\nALL OUTPUTS:")
    print(f"  data:       {DATA_DIR}/")
    print(f"  checkpoints:{CKPT_DIR}/")
    print(f"  per-run:    {RUN_PDF_DIR}/  ({total} PDFs)")
    print(f"  summary:    {OUT_DIR / 'summary.pdf'}")
    print(f"  json:       {results_path}")


if __name__ == '__main__':
    main()

[init] device = cuda
[init] gpu   = NVIDIA RTX PRO 6000 Blackwell Server Edition
[init] mem   = 102.0 GB

GEN5 H100 STRESS SWEEP — CWRU BEARING FAULT
Sizes: [32, 64, 128]  ·  Seeds: 20  ·  Steps: 2000
Total runs: 60
[data] normal: downloading from https://engineering.case.edu/sites/default/files/97.mat
[data] normal: 3811 KB
[data] inner: downloading from https://engineering.case.edu/sites/default/files/105.mat
[data] inner: 2842 KB
[data] outer: downloading from https://engineering.case.edu/sites/default/files/130.mat
[data] outer: 2859 KB
[data] ball: downloading from https://engineering.case.edu/sites/default/files/118.mat
[data] ball: 2873 KB
[data] normal: 243938 samples (20.3s @ 12kHz)
[data] inner: 121265 samples (10.1s @ 12kHz)
[data] outer: 121991 samples (10.2s @ 12kHz)
[data] ball: 122571 samples (10.2s @ 12kHz)
[data] windows: 948 train, 237 test, 4 classes
[1/60] RUN   H=32 seed=0  